# Direct and residual path-loss prediction
**Objective:** fit a planning baseline, direct learner and residual correction with the same allowed inputs.
**Prerequisites:** notebooks 00–01. The target follows the Bonn snapshot convention, not a universal radio identity.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## 1. Audit the target and the information boundary

In [ ]:
from experiments import load_pathloss
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,root_mean_squared_error

d,features=load_pathloss()
assert not {'rssi','snr','target'} & set(features)
np.testing.assert_allclose(d.target,15-(d.rssi+np.minimum(d.snr,0)))
X=d[features].to_numpy();y=d.target.to_numpy()
tr,te=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,groups=d.spatial_group))
assert not set(d.spatial_group.iloc[tr]) & set(d.spatial_group.iloc[te])
display(d[features+['target']].head())


## 2. Construct residual labels explicitly
For baseline 120 dB and observation 116 dB, the residual is −4 dB. At prediction time, add the learned correction to the baseline. Both components fit only training examples.

In [ ]:
baseline=LinearRegression().fit(X[tr,:1],y[tr])
training_residual=y[tr]-baseline.predict(X[tr,:1])
display(pd.DataFrame({'truth':y[tr][:5],'baseline':baseline.predict(X[tr,:1])[:5],'residual':training_residual[:5]}))
def forest():
    return RandomForestRegressor(n_estimators=80,min_samples_leaf=5,random_state=42,n_jobs=2)
direct=forest().fit(X[tr],y[tr]);correction=forest().fit(X[tr],training_residual)
predictions={'log-distance':baseline.predict(X[te,:1]),'direct RF':direct.predict(X[te]),'residual RF':baseline.predict(X[te,:1])+correction.predict(X[te])}
display(pd.DataFrame({name:{'MAE_dB':mean_absolute_error(y[te],pred),'RMSE_dB':root_mean_squared_error(y[te],pred)} for name,pred in predictions.items()}).T)


## 3. Diagnose the error and compare information

In [ ]:
fig,ax=plt.subplots(figsize=(6,3))
ax.scatter(X[te,0],y[te]-predictions['direct RF'],s=3,alpha=.25)
ax.axhline(0,color='black',linewidth=.8);ax.set(xlabel='log10 distance (km)',ylabel='Residual (dB)');plt.show()
m=pd.read_csv(GUIDE/'metrics.csv')
display(m[m.case.str.startswith('environment_')].groupby(['case','model']).rmse.agg(['mean','std']))


**Exercise:** does adding weather and obtaining lower error prove a causal weather effect? Does a hybrid automatically extrapolate better?

<details><summary>Feedback after your attempt</summary>

these are observational conditional predictions. Weather can proxy for time or other conditions; the ablation isolates an input set, not a causal intervention. A residual learner can extrapolate poorly even when its baseline has a physical interpretation.

</details>

**Adaptation task:** state the actual transmit power, gains/losses and source conventions before constructing a new target. Hold out the sites or gateways corresponding to your intended planning use.